# [7.4] Mini Natural Language Autoencoders - Solutions

Reference validation notebook for the section-local mini NLA implementation.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter7_activation_to_language"
section = "part4_mini_natural_language_autoencoders"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_mini_natural_language_autoencoders.tests as tests
from chapter7_activation_to_language.exercises.part4_mini_natural_language_autoencoders import solutions

In [ ]:
tests.test_build_nla_training_batch_validates_alignment(
    solutions.build_nla_training_batch,
)
tests.test_generated_explanations_do_not_hide_numeric_coefficients(
    solutions._numeric_literal_count,
)
tests.test_activation_reconstruction_report_beats_text_only_baseline(
    solutions.activation_reconstruction_report,
)
tests.test_logit_diff_preservation_report_checks_actual_logit_diff(
    solutions.logit_diff_preservation_report,
)
tests.test_latent_preservation_report_requires_accuracy_and_agreement(
    solutions.latent_preservation_report,
)
tests.test_brevity_and_counterfactual_reports_reject_prompt_copying(
    solutions.generated_text_brevity_report,
    solutions.counterfactual_explanation_report,
)
tests.test_trainable_discrete_bottleneck_learns_phrase_ids(
    solutions.train_discrete_nla_bottleneck,
)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["reconstruction"]["beats_text_only"]
assert 0.149 < contract["logit_diff"]["mean_abs_error"] < 0.151
assert contract["latent_preservation"]["preserves_latents"]
assert contract["brevity"]["shorter_than_original"]
assert contract["counterfactual"]["explanation_changed"]
assert contract["trainable_bottleneck"]["eval_phrase_accuracy"] == 1.0
assert contract["trainable_bottleneck"]["beats_blank_text"]
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["activation_shape"] == [8, 512]
assert gpu["text_bottleneck"] == "discrete_natural_language_phrase_bottleneck"
assert gpu["live_trainable_nla"]
assert not gpu["report_replay"]
assert gpu["trainable_encoder_parameter_count"] > 0
assert gpu["trainable_decoder_parameter_count"] > 0
assert gpu["trainable_encoder_train_accuracy"] == 1.0
assert gpu["trainable_eval_phrase_accuracy"] >= 0.75
assert gpu["trainable_encoder_final_loss"] < 0.01
assert gpu["trainable_beats_blank_text"]
assert gpu["phrase_count"] == 12
assert gpu["numeric_literal_count"] == 0
assert gpu["activation_mse"] < gpu["text_only_mse"]
assert gpu["activation_mse"] < gpu["prompt_label_baseline_mse"]
assert gpu["mean_cosine_similarity"] >= 0.93
assert gpu["preserves_target_logit_diff"]
assert gpu["probe_logit_mean_abs_error"] <= 2.0
assert gpu["preserves_latents"]
assert gpu["text_only_prediction_accuracy"] == 0.5
assert gpu["nla_prediction_accuracy"] == 1.0
assert gpu["passes_ood"]
assert gpu["counterfactual_explanation_changed"]
assert gpu["shuffled_control_worse"]
assert gpu["blank_text_control_worse"]
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "model_name",
    "text_bottleneck",
    "example_generated_explanation",
    "numeric_literal_count",
    "activation_mse",
    "text_only_mse",
    "prompt_label_baseline_mse",
    "mean_cosine_similarity",
    "probe_logit_mean_abs_error",
    "nla_prediction_accuracy",
    "text_only_prediction_accuracy",
    "counterfactual_explanation_changed",
    "blank_text_mse",
    "peak_vram_gb",
]}